# RAG 팀과제: 카페 Kiosk 자연어 인터페이스

**작성:** 구범용  
**목표:** 실제 카페와 유사한 메뉴 문서를 검색(Retrieval)하고, 검색된 근거만 Claude에 제공해
메뉴 안내·추천·옵션·비교·주문을 처리하는 자연어 Kiosk를 구현한다.

## 요구사항 대응

- 음료·디저트 설명, 가격, 사이즈, 샷·시럽 옵션 제공
- 취향 기반 추천과 식사 메뉴 페어링
- 옵션을 포함한 주문 추가·변경·삭제
- 여러 메뉴의 맛·카페인·가격 비교
- 카페 범위 밖 질문과 검색 결과 없음의 안전한 처리
- HTI와 대표 시연 시나리오 제공
- 전체 15개 메뉴와 옵션 추가금을 시작 화면에서 확인
- ‘네’·‘둘 다’ 등 짧은 응답에서 최근 사용자 메뉴 문맥 유지
- 동일 옵션 주문 수량 병합, 일부 취소, 서버 기준 가격·합계 재계산
- 이름과 내용만 입력하는 피드백 로그와 상태 이력·내보내기 제공


In [1]:
import os
from getpass import getpass
def resolve_secret(name):
    value = os.getenv(name, "").strip()
    if value:
        return value
    return getpass(f"{name} 입력(화면에 표시되지 않음): ").strip()


from anthropic import Anthropic

ANTHROPIC_API_KEY = resolve_secret("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise RuntimeError("ANTHROPIC_API_KEY가 필요합니다.")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-haiku-5-5")
client = Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic API 키를 메모리에 불러왔습니다. 키 값은 출력하지 않습니다.")


Anthropic API 키를 메모리에 불러왔습니다. 키 값은 출력하지 않습니다.


## 1. RAG 메뉴 문서

각 문서는 설명·가격·옵션·추천 태그를 포함한다. 가격 단위는 원이다.


In [2]:
MENU_CATALOG = [
    {"id":"americano","name":"아메리카노","category":"커피","description":"진한 에스프레소와 물, 깔끔하고 쌉쌀함","prices":{"S":4000,"M":4500,"L":5000},"caffeine":"높음","temperature":["hot","ice"],"options":["샷 추가","바닐라 시럽"],"tags":["진한","깔끔한","저당"]},
    {"id":"cafe_latte","name":"카페라떼","category":"커피","description":"에스프레소와 우유, 부드럽고 고소함","prices":{"S":4700,"M":5200,"L":5700},"caffeine":"중간","temperature":["hot","ice"],"options":["샷 추가","바닐라 시럽","오트밀크 변경"],"tags":["부드러운","고소한"]},
    {"id":"cappuccino","name":"카푸치노","category":"커피","description":"에스프레소와 풍성한 우유 거품, 라떼보다 커피 향이 선명함","prices":{"S":4700,"M":5200},"caffeine":"중간","temperature":["hot"],"options":["샷 추가","시나몬"],"tags":["진한","거품","고소한"]},
    {"id":"vanilla_latte","name":"바닐라라떼","category":"커피","description":"카페라떼에 바닐라 시럽을 더한 달콤한 음료","prices":{"S":5200,"M":5700,"L":6200},"caffeine":"중간","temperature":["hot","ice"],"options":["샷 추가","시럽 적게","오트밀크 변경"],"tags":["달콤한","부드러운"]},
    {"id":"caramel_macchiato","name":"카라멜마키아토","category":"커피","description":"바닐라 우유와 에스프레소, 카라멜 드리즐의 진한 단맛","prices":{"S":5500,"M":6000,"L":6500},"caffeine":"중간","temperature":["hot","ice"],"options":["샷 추가","카라멜 적게","오트밀크 변경"],"tags":["매우 달콤한","디저트 음료"]},
    {"id":"cold_brew","name":"콜드브루","category":"커피","description":"저온 추출로 산미가 적고 묵직하며 깔끔함","prices":{"M":5000,"L":5500},"caffeine":"높음","temperature":["ice"],"options":["바닐라 시럽","오트밀크 추가"],"tags":["진한","깔끔한","산미 적은"]},
    {"id":"decaf_americano","name":"디카페인 아메리카노","category":"커피","description":"카페인을 줄인 깔끔한 아메리카노","prices":{"S":4500,"M":5000,"L":5500},"caffeine":"매우 낮음","temperature":["hot","ice"],"options":["샷 추가","바닐라 시럽"],"tags":["디카페인","깔끔한"]},
    {"id":"matcha_latte","name":"말차라떼","category":"논커피","description":"제주 말차와 우유의 쌉싸름하고 달콤한 조화","prices":{"S":5200,"M":5700,"L":6200},"caffeine":"낮음","temperature":["hot","ice"],"options":["당도 적게","오트밀크 변경"],"tags":["달콤한","쌉싸름한","논커피"]},
    {"id":"chocolate","name":"초콜릿라떼","category":"논커피","description":"다크 초콜릿과 우유로 만든 진하고 달콤한 음료","prices":{"S":5000,"M":5500,"L":6000},"caffeine":"매우 낮음","temperature":["hot","ice"],"options":["휘핑 추가","당도 적게","오트밀크 변경"],"tags":["달콤한","초콜릿","논커피"]},
    {"id":"yuzu_tea","name":"유자차","category":"티","description":"유자청의 상큼하고 달콤한 향이 나는 차","prices":{"S":4500,"M":5000},"caffeine":"없음","temperature":["hot","ice"],"options":["당도 적게"],"tags":["상큼한","무카페인"]},
    {"id":"earl_grey","name":"얼그레이 티","category":"티","description":"베르가못 향의 깔끔한 홍차","prices":{"S":4200,"M":4700},"caffeine":"낮음","temperature":["hot","ice"],"options":["레몬 추가","시럽 추가"],"tags":["향긋한","깔끔한"]},
    {"id":"plain_croissant","name":"플레인 크루아상","category":"디저트","description":"버터 풍미와 바삭한 결의 기본 크루아상","prices":{"ONE":3800},"caffeine":"없음","temperature":["room"],"options":["데우기"],"tags":["고소한","빵"]},
    {"id":"chocolate_cake","name":"초콜릿 케이크","category":"디저트","description":"진한 다크 초콜릿 크림의 촉촉한 케이크","prices":{"ONE":6500},"caffeine":"매우 낮음","temperature":["cold"],"options":[],"tags":["달콤한","케이크"]},
    {"id":"cheesecake","name":"뉴욕 치즈케이크","category":"디저트","description":"꾸덕한 크림치즈와 은은한 산미의 케이크","prices":{"ONE":6200},"caffeine":"없음","temperature":["cold"],"options":[],"tags":["고소한","케이크"]},
    {"id":"vegan_cookie","name":"비건 오트 쿠키","category":"디저트","description":"버터와 달걀 없이 만든 오트밀 쿠키","prices":{"ONE":3200},"caffeine":"없음","temperature":["room"],"options":[],"tags":["비건","쿠키","담백한"]},
]

MEAL_PAIRING_GUIDES = [
    {"label":"샌드위치","aliases":["샌드위치","토스트","햄버거"],"recommendations":[("americano","깔끔하고 쌉쌀해 빵과 속재료의 풍미를 정리해 줍니다."),("cafe_latte","부드럽고 고소해 담백한 샌드위치와 잘 어울립니다."),("earl_grey","베르가못 향이 산뜻해 채소나 햄이 든 샌드위치와 균형을 이룹니다.")]},
    {"label":"샐러드","aliases":["샐러드","채소식","야채식"],"recommendations":[("earl_grey","향긋하고 깔끔해 신선한 채소 맛을 해치지 않습니다."),("yuzu_tea","상큼한 유자 향이 가벼운 샐러드와 잘 어울립니다."),("cold_brew","산미가 적고 깔끔해 고소한 드레싱과 균형을 이룹니다.")]},
    {"label":"파스타","aliases":["파스타","스파게티","면요리"],"recommendations":[("americano","쌉쌀하고 깔끔해 크림이나 치즈의 묵직함을 정리해 줍니다."),("cold_brew","산미가 적고 묵직해 진한 소스와 무난하게 어울립니다."),("yuzu_tea","상큼한 향이 토마토소스나 오일 파스타 뒤 입안을 산뜻하게 합니다.")]},
    {"label":"치킨","aliases":["치킨","닭요리","닭고기"],"recommendations":[("yuzu_tea","상큼한 유자 향이 기름진 맛을 산뜻하게 정리해 줍니다."),("cold_brew","깔끔하고 진해 튀김이나 양념의 강한 맛과 균형을 이룹니다."),("americano","단맛이 적고 쌉쌀해 치킨의 풍미를 방해하지 않습니다.")]},
]

OPTION_PRICES = {"샷 추가":700,"바닐라 시럽":500,"오트밀크 변경":700,"오트밀크 추가":700,"휘핑 추가":500,"레몬 추가":300,"시럽 추가":500}

print(f"메뉴 문서 {len(MENU_CATALOG)}개를 준비했습니다.")


메뉴 문서 15개를 준비했습니다.


## 2. 검색기(Retriever)

한국어 띄어쓰기 변화에 강한 문자 n-gram TF-IDF를 표준 라이브러리만으로 구현해 메뉴 문서를 검색한다.


In [3]:
from collections import Counter
from math import log, sqrt


def menu_document(item):
    prices = " ".join(f"{size} {price}원" for size, price in item["prices"].items())
    pairing_terms = []
    for guide in MEAL_PAIRING_GUIDES:
        if any(menu_id == item["id"] for menu_id, _ in guide["recommendations"]):
            pairing_terms.extend([guide["label"], *guide["aliases"]])
    return " ".join([
        item["name"], item["category"], item["description"], prices,
        "카페인 " + item["caffeine"], " ".join(item["temperature"]),
        " ".join(item["options"]), " ".join(item["tags"]), " ".join(pairing_terms),
    ])


MENU_DOCUMENTS = [menu_document(item) for item in MENU_CATALOG]


def character_ngrams(text, minimum=2, maximum=4):
    normalized = " ".join(str(text).lower().split())
    padded = f" {normalized} "
    return Counter(
        padded[index:index + size]
        for size in range(minimum, maximum + 1)
        for index in range(max(0, len(padded) - size + 1))
    )


DOCUMENT_TERMS = [character_ngrams(document) for document in MENU_DOCUMENTS]
DOCUMENT_FREQUENCY = Counter()
for terms in DOCUMENT_TERMS:
    DOCUMENT_FREQUENCY.update(terms.keys())
DOCUMENT_COUNT = len(DOCUMENT_TERMS)
IDF = {
    term: log((1 + DOCUMENT_COUNT) / (1 + frequency)) + 1
    for term, frequency in DOCUMENT_FREQUENCY.items()
}


def tfidf_vector(terms):
    total = sum(terms.values()) or 1
    return {
        term: (count / total) * IDF.get(term, log(1 + DOCUMENT_COUNT) + 1)
        for term, count in terms.items()
    }


MENU_VECTORS = [tfidf_vector(terms) for terms in DOCUMENT_TERMS]


def cosine_score(left, right):
    numerator = sum(value * right.get(term, 0.0) for term, value in left.items())
    left_norm = sqrt(sum(value * value for value in left.values()))
    right_norm = sqrt(sum(value * value for value in right.values()))
    return numerator / (left_norm * right_norm) if left_norm and right_norm else 0.0


def retrieve_menu(query, k=5, minimum_score=0.02):
    query_vector = tfidf_vector(character_ngrams(str(query)))
    normalized_query = " ".join(str(query).lower().split())
    scores = [
        cosine_score(query_vector, menu_vector) + (1.0 if item["name"].lower() in normalized_query else 0.0)
        for item, menu_vector in zip(MENU_CATALOG, MENU_VECTORS)
    ]
    ranked = sorted(enumerate(scores), key=lambda pair: pair[1], reverse=True)
    return [
        {"score": round(float(score), 4), "menu": MENU_CATALOG[index]}
        for index, score in ranked[:k] if score >= minimum_score
    ]


for example in ["달콤한 음료", "아메리카노와 카푸치노 가격", "비건 디저트", "샌드위치와 어울리는 음료"]:
    print(example, "->", [(row["menu"]["name"], row["score"]) for row in retrieve_menu(example, 3)])


달콤한 음료 -> [('바닐라라떼', 0.3177), ('초콜릿라떼', 0.262), ('카라멜마키아토', 0.1455)]
아메리카노와 카푸치노 가격 -> [('카푸치노', 1.1268), ('아메리카노', 1.0888), ('디카페인 아메리카노', 0.2095)]
비건 디저트 -> [('비건 오트 쿠키', 0.3045), ('뉴욕 치즈케이크', 0.0734), ('플레인 크루아상', 0.0712)]
샌드위치와 어울리는 음료 -> [('카페라떼', 0.1206), ('얼그레이 티', 0.1065), ('아메리카노', 0.1006)]


## 3. HTI(Hierarchical Task Inventory)

시연과 검증은 아래 하위 과업을 기준으로 구성한다.


In [4]:
HTI = {
    "1_탐색": ["메뉴 목록", "메뉴 설명", "가격·사이즈", "옵션·카페인"],
    "2_추천": ["맛 선호", "카페인 선호", "식이 제한", "음료·디저트 조합"],
    "3_비교": ["두 메뉴 맛 비교", "가격 비교", "카페인 비교"],
    "4_주문": ["메뉴 추가", "수량", "사이즈", "온도", "옵션 변경", "삭제", "장바구니 확인"],
    "5_복구": ["모호한 요청 재질문", "없는 메뉴", "지원하지 않는 옵션", "범위 밖 질문"],
}

for category, tasks in HTI.items():
    print(category, ":", " · ".join(tasks))


1_탐색 : 메뉴 목록 · 메뉴 설명 · 가격·사이즈 · 옵션·카페인
2_추천 : 맛 선호 · 카페인 선호 · 식이 제한 · 음료·디저트 조합
3_비교 : 두 메뉴 맛 비교 · 가격 비교 · 카페인 비교
4_주문 : 메뉴 추가 · 수량 · 사이즈 · 온도 · 옵션 변경 · 삭제 · 장바구니 확인
5_복구 : 모호한 요청 재질문 · 없는 메뉴 · 지원하지 않는 옵션 · 범위 밖 질문


## 4. RAG Kiosk

검색된 메뉴만 컨텍스트로 제공하고, 모델의 장바구니 동작은 카탈로그와 옵션 목록으로 다시 검증한다.


In [5]:
import json


KIOSK_SYSTEM = """
당신은 카페 자연어 Kiosk입니다. 검색 컨텍스트에 있는 메뉴 정보만 사용하세요.
메뉴 설명·가격·옵션 질문에는 정확한 근거로 답하고, 선호 질문에는 최대 3개를 이유와 함께 추천하세요.
복합 질문은 각 부분을 빠짐없이 답하세요. 범위 밖 질문은 정중히 카페 업무로 돌려보내세요.
샌드위치·샐러드·파스타·치킨은 판매 메뉴가 아니라 음료 추천을 위한 식사 페어링 조건일 수 있습니다.
추천 대상으로는 검색 컨텍스트에 포함된 실제 판매 메뉴만 사용하고, 식사 이름을 판매 메뉴라고 말하지 마세요.
주문 정보가 모호하거나 옵션이 지원되지 않으면 추측하지 말고 하나의 확인 질문을 하세요.
응답은 반드시 JSON 객체 하나이며 answer, intent, cited_menu_ids, cart_operation, cart_item을 포함합니다.
intent는 inform, recommend, compare, order, clarify, out_of_scope 중 하나입니다.
cart_operation은 none, add, replace, remove, clear 중 하나입니다.
cart_item은 item_id, size, temperature, options, quantity를 포함하며 주문 동작이 없으면 빈 값과 기본 수량을 사용합니다.
""".strip()


KIOSK_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "intent": {"type": "string", "enum": ["inform", "recommend", "compare", "order", "clarify", "out_of_scope"]},
        "cited_menu_ids": {"type": "array", "items": {"type": "string"}},
        "cart_operation": {"type": "string", "enum": ["none", "add", "replace", "remove", "clear"]},
        "cart_item": {
            "type": "object",
            "properties": {
                "item_id": {"type": "string"},
                "size": {"type": "string"},
                "temperature": {"type": "string"},
                "options": {"type": "array", "items": {"type": "string"}},
                "quantity": {"type": "integer"},
            },
            "required": ["item_id", "size", "temperature", "options", "quantity"],
            "additionalProperties": False,
        },
    },
    "required": ["answer", "intent", "cited_menu_ids", "cart_operation", "cart_item"],
    "additionalProperties": False,
}


PAIRING_GENERIC_WORDS = ("식사 메뉴", "식사메뉴", "음식과 어울", "식사와 어울", "페어링", "같이 먹", "곁들")
PAIRING_ORDER_WORDS = ("주문", "담아", "장바구니", "결제", "살래", "주세요")


def matching_meal_pairings(text):
    normalized = "".join(str(text).lower().split())
    return [guide for guide in MEAL_PAIRING_GUIDES if any("".join(alias.lower().split()) in normalized for alias in guide["aliases"])]


def build_pairing_response(text, history, cart):
    matched = matching_meal_pairings(text)
    previous = history[-1] if history else {}
    prior_question = previous.get("role") == "assistant" and "함께 드실 식사를 말씀해 주세요" in previous.get("content", "")
    asks_pairing = matched or prior_question or any(word in text for word in PAIRING_GENERIC_WORDS)
    if not asks_pairing or any(word in text for word in PAIRING_ORDER_WORDS):
        return None
    total = sum(row["unit_price"] * row["quantity"] for row in cart)
    if not matched:
        return {"answer":"판매 중인 음료와 디저트 안에서 식사 페어링을 추천해 드릴게요. 샌드위치·샐러드·파스타·치킨 중 함께 드실 식사를 말씀해 주세요.","intent":"clarify","cited_menu_ids":[],"retrieved":[],"cart":list(cart),"cart_total":total}
    selected = []
    for guide in matched:
        for menu_id, reason in guide["recommendations"]:
            if any(row["menu"]["id"] == menu_id for row in selected):
                continue
            menu = next((item for item in MENU_CATALOG if item["id"] == menu_id), None)
            if menu:
                selected.append({"menu":menu,"reason":reason})
            if len(selected) == 3:
                break
        if len(selected) == 3:
            break
    meal_names = "·".join(dict.fromkeys(guide["label"] for guide in matched))
    price_label = lambda menu: " / ".join(f"{size} {price:,}원" for size, price in menu["prices"].items())
    lines = [f"{index}. {row['menu']['name']} ({price_label(row['menu'])}): {row['reason']}" for index, row in enumerate(selected, 1)]
    return {"answer":f"{meal_names}에 곁들일 음료로는 현재 판매 중인 메뉴에서 다음을 추천합니다.\n" + "\n".join(lines),"intent":"recommend","cited_menu_ids":[row["menu"]["id"] for row in selected],"retrieved":[{"id":row["menu"]["id"],"name":row["menu"]["name"],"price_label":price_label(row["menu"]),"score":1.0} for row in selected],"cart":list(cart),"cart_total":total}


class CafeKioskRAG:
    def __init__(self, model=MODEL):
        self.model = model
        self.history = []
        self.cart = []

    def reset(self):
        self.history.clear()
        self.cart.clear()

    def _menu_by_id(self, item_id):
        return next((item for item in MENU_CATALOG if item["id"] == item_id), None)

    def _apply_cart(self, operation, cart_item):
        if operation == "none" or not cart_item:
            return
        if operation == "clear":
            self.cart.clear()
            return
        item = self._menu_by_id(str(cart_item.get("item_id", "")))
        if not item:
            raise ValueError("카탈로그에 없는 메뉴입니다.")
        size = str(cart_item.get("size") or next(iter(item["prices"])))
        if size not in item["prices"]:
            raise ValueError("지원하지 않는 사이즈입니다.")
        temperature = str(cart_item.get("temperature") or item["temperature"][0])
        if temperature not in item["temperature"]:
            raise ValueError("지원하지 않는 온도 옵션입니다.")
        options = [str(value) for value in cart_item.get("options") or []]
        if any(value not in item["options"] for value in options):
            raise ValueError("지원하지 않는 추가 옵션입니다.")
        quantity = max(1, min(10, int(cart_item.get("quantity") or 1)))
        row = {"item_id":item["id"],"name":item["name"],"size":size,"temperature":temperature,"options":options,"quantity":quantity}
        row["unit_price"] = item["prices"][size] + sum(OPTION_PRICES.get(option, 0) for option in options)
        if operation == "remove":
            for index in range(len(self.cart) - 1, -1, -1):
                if self.cart[index]["item_id"] == item["id"]:
                    self.cart[index]["quantity"] -= quantity
                    if self.cart[index]["quantity"] <= 0:
                        self.cart.pop(index)
                    break
        elif operation == "replace":
            self.cart = [existing for existing in self.cart if existing["item_id"] != item["id"]]
            self.cart.append(row)
        elif operation == "add":
            existing = next((value for value in self.cart if value["item_id"] == row["item_id"] and value["size"] == row["size"] and value["temperature"] == row["temperature"] and value["options"] == row["options"]), None)
            if existing:
                existing["quantity"] = min(10, existing["quantity"] + quantity)
            else:
                self.cart.append(row)

    def ask(self, user_text):
        text = str(user_text).strip()
        if not text:
            return {"answer":"주문하거나 궁금한 메뉴를 말씀해 주세요.","intent":"clarify","retrieved":[],"cart":self.cart}
        if text.lower() in {"exit", "quit", "종료", "/종료"}:
            return {"answer":"주문 대화를 종료합니다.","intent":"ended","retrieved":[],"cart":self.cart}
        compact = "".join(text.split()).rstrip(".!~")
        if compact in {"네","예","응","그래","좋아","그걸로","네주문할게요","주문할게요"} and self.cart:
            total = sum(row["unit_price"] * row["quantity"] for row in self.cart)
            answer = f"현재 장바구니 내용으로 주문을 확정했습니다. 총 {total:,}원입니다."
            self.history.extend([{"role":"user","content":text},{"role":"assistant","content":answer}])
            return {"answer":answer,"intent":"order","order_completed":True,"retrieved":[],"cart":list(self.cart),"cart_total":total}
        pairing_result = build_pairing_response(text, self.history, self.cart)
        if pairing_result:
            self.history.extend([{"role":"user","content":text},{"role":"assistant","content":pairing_result["answer"]}])
            return pairing_result
        short_replies = {"네","예","응","그래","좋아","둘다","그걸로","그거","둘모두"}
        recent_user = next((entry["content"] for entry in reversed(self.history) if entry.get("role") == "user"), "")
        search_text = f"{recent_user} {text}".strip() if compact in short_replies else text
        retrieved = retrieve_menu(search_text, k=5)
        context = [row["menu"] for row in retrieved]
        prompt = {
            "conversation": self.history[-6:],
            "current_cart": self.cart,
            "retrieved_menu_documents": context,
            "option_prices": OPTION_PRICES,
            "user_request": text,
        }
        response = client.messages.create(
            model=self.model,
            max_tokens=900,
            system=KIOSK_SYSTEM,
            messages=[{"role":"user","content":json.dumps(prompt, ensure_ascii=False)}],
            output_config={"format": {"type": "json_schema", "schema": KIOSK_RESPONSE_SCHEMA}},
        )
        output_text = "\n".join(block.text for block in response.content if block.type == "text").strip()
        try:
            result = json.loads(output_text)
        except Exception:
            result = {"answer":output_text or "요청을 처리하지 못했습니다.","intent":"clarify","cited_menu_ids":[],"cart_operation":"none","cart_item":None}
        try:
            self._apply_cart(result.get("cart_operation", "none"), result.get("cart_item"))
        except ValueError as error:
            result["intent"] = "clarify"
            result["answer"] = str(error) + " 가능한 옵션을 다시 선택해 주세요."
        self.history.extend([
            {"role":"user","content":text},
            {"role":"assistant","content":result.get("answer", "")},
        ])
        return {
            **result,
            "retrieved": [{"id":row["menu"]["id"],"name":row["menu"]["name"],"price_label":" / ".join(f"{size} {price:,}원" for size, price in row["menu"]["prices"].items()),"score":row["score"]} for row in retrieved],
            "cart":list(self.cart),
            "cart_total":sum(row["unit_price"] * row["quantity"] for row in self.cart),
        }


## 5. 대표 시연

정보·추천·복합 비교·옵션 주문·범위 밖 질문을 순서대로 점검한다.


In [6]:
RUN_LIVE_DEMO = True

kiosk = CafeKioskRAG()
demo_requests = [
    "아메리카노 사이즈별 가격과 카페인을 알려줘.",
    "달콤하지만 커피가 아닌 음료를 추천해줘.",
    "식사 메뉴와 어울리는 음료를 추천해줘.",
    "샌드위치",
    "치킨",
    "아메리카노랑 카푸치노 중 뭐가 더 진하고 가격은 어떻게 달라?",
    "아이스 카페라떼 M에 바닐라 시럽 추가해서 한 잔 담아줘.",
    "오늘 주식 시장 전망을 알려줘.",
]

if RUN_LIVE_DEMO:
    for request in demo_requests:
        result = kiosk.ask(request)
        print("사용자:", request)
        print("Kiosk:", result["answer"])
        print("의도:", result["intent"], "검색:", result["retrieved"])
        print("장바구니:", result["cart"], "합계:", result["cart_total"], "원\n")
else:
    print("RUN_LIVE_DEMO=True로 바꾸면 실제 RAG+Claude 시연을 실행합니다.")


사용자: 아메리카노 사이즈별 가격과 카페인을 알려줘.
Kiosk: 아메리카노는 S 4,000원, M 4,500원, L 5,000원이며 카페인은 높음입니다. 참고로 디카페인 아메리카노는 S 4,500원, M 5,000원, L 5,500원이고 카페인은 매우 낮음입니다.
의도: inform 검색: [{'id': 'americano', 'name': '아메리카노', 'score': 1.1037}, {'id': 'decaf_americano', 'name': '디카페인 아메리카노', 'score': 0.2361}]
장바구니: [] 합계: 0 원



사용자: 달콤하지만 커피가 아닌 음료를 추천해줘.
Kiosk: 달콤하면서 커피가 아닌 음료로는 초콜릿라떼를 추천드려요. 다크 초콜릿과 우유로 만든 진하고 달콤한 논커피 음료이며, 카페인은 매우 낮습니다. 가격은 S 5,000원, M 5,500원, L 6,000원이고, 따뜻하게(hot) 또는 차갑게(ice) 드실 수 있습니다. 당도를 줄이는 '당도 적게', 휘핑을 올리는 '휘핑 추가', '오트밀크 변경' 옵션을 선택하실 수 있어요. 현재 검색된 메뉴 중 논커피 달콤한 음료는 이 하나입니다.
의도: recommend 검색: [{'id': 'vanilla_latte', 'name': '바닐라라떼', 'score': 0.0441}, {'id': 'chocolate', 'name': '초콜릿라떼', 'score': 0.034}, {'id': 'caramel_macchiato', 'name': '카라멜마키아토', 'score': 0.0289}, {'id': 'decaf_americano', 'name': '디카페인 아메리카노', 'score': 0.0219}]
장바구니: [] 합계: 0 원



사용자: 식사 메뉴와 어울리는 음료를 추천해줘.
Kiosk: 판매 중인 음료와 디저트 안에서 식사 페어링을 추천해 드릴게요. 샌드위치·샐러드·파스타·치킨 중 함께 드실 식사를 말씀해 주세요.
의도: clarify 검색: []
장바구니: [] 합계: 0 원



사용자: 샌드위치
Kiosk: 샌드위치에 곁들일 음료로는 현재 판매 중인 메뉴에서 다음을 추천합니다.
1. 아메리카노: 깔끔하고 쌉쌀해 빵과 속재료의 풍미를 정리해 줍니다.
2. 카페라떼: 부드럽고 고소해 담백한 샌드위치와 잘 어울립니다.
3. 얼그레이 티: 베르가못 향이 산뜻해 채소나 햄이 든 샌드위치와 균형을 이룹니다.
의도: recommend 검색: [{'id': 'americano', 'name': '아메리카노', 'score': 1.0}, {'id': 'cafe_latte', 'name': '카페라떼', 'score': 1.0}, {'id': 'earl_grey', 'name': '얼그레이 티', 'score': 1.0}]
장바구니: [] 합계: 0 원



사용자: 치킨
Kiosk: 치킨에 곁들일 음료로는 현재 판매 중인 메뉴에서 다음을 추천합니다.
1. 유자차: 상큼한 유자 향이 기름진 맛을 산뜻하게 정리해 줍니다.
2. 콜드브루: 깔끔하고 진해 튀김이나 양념의 강한 맛과 균형을 이룹니다.
3. 아메리카노: 단맛이 적고 쌉쌀해 치킨의 풍미를 방해하지 않습니다.
의도: recommend 검색: [{'id': 'yuzu_tea', 'name': '유자차', 'score': 1.0}, {'id': 'cold_brew', 'name': '콜드브루', 'score': 1.0}, {'id': 'americano', 'name': '아메리카노', 'score': 1.0}]
장바구니: [] 합계: 0 원



사용자: 아메리카노랑 카푸치노 중 뭐가 더 진하고 가격은 어떻게 달라?
Kiosk: 두 메뉴 모두 '진한' 태그가 있지만 성격이 다릅니다. 아메리카노는 진한 에스프레소와 물로 만들어 깔끔하고 쌉쌀하며 카페인이 '높음'이고, 카푸치노는 에스프레소에 풍성한 우유 거품을 더해 커피 향이 선명하고 고소하며 카페인은 '중간'입니다. 카페인 기준으로는 아메리카노가 더 강합니다. 가격은 아메리카노가 S 4,000원, M 4,500원, L 5,000원이고, 카푸치노는 S 4,700원, M 5,200원으로 S와 M 모두 700원 더 비쌉니다. 카푸치노는 L 사이즈가 없고 따뜻하게(hot)만 제공됩니다.
의도: compare 검색: [{'id': 'americano', 'name': '아메리카노', 'score': 1.0872}, {'id': 'cappuccino', 'name': '카푸치노', 'score': 1.0824}, {'id': 'decaf_americano', 'name': '디카페인 아메리카노', 'score': 0.1282}, {'id': 'chocolate', 'name': '초콜릿라떼', 'score': 0.0481}, {'id': 'vanilla_latte', 'name': '바닐라라떼', 'score': 0.0212}]
장바구니: [] 합계: 0 원



사용자: 아이스 카페라떼 M에 바닐라 시럽 추가해서 한 잔 담아줘.
Kiosk: 아이스 카페라떼 M에 바닐라 시럽을 추가해 한 잔 담았습니다. 금액은 카페라떼 M 5,200원과 바닐라 시럽 500원을 합해 5,700원입니다.
의도: order 검색: [{'id': 'cafe_latte', 'name': '카페라떼', 'score': 1.1162}, {'id': 'vanilla_latte', 'name': '바닐라라떼', 'score': 0.1585}, {'id': 'decaf_americano', 'name': '디카페인 아메리카노', 'score': 0.0709}, {'id': 'americano', 'name': '아메리카노', 'score': 0.0697}, {'id': 'cold_brew', 'name': '콜드브루', 'score': 0.0618}]
장바구니: [{'item_id': 'cafe_latte', 'name': '카페라떼', 'size': 'M', 'temperature': 'ice', 'options': ['바닐라 시럽'], 'quantity': 1, 'unit_price': 5700}] 합계: 5700 원



사용자: 오늘 주식 시장 전망을 알려줘.
Kiosk: 죄송하지만 주식 시장 전망은 카페 주문과 메뉴 안내 범위 밖이라 답해드리기 어렵습니다. 대신 메뉴 선택이나 주문을 도와드릴 수 있어요. 현재 장바구니에는 아이스 카페라떼 M(바닐라 시럽 추가) 1잔이 담겨 있습니다.
의도: out_of_scope 검색: []
장바구니: [{'item_id': 'cafe_latte', 'name': '카페라떼', 'size': 'M', 'temperature': 'ice', 'options': ['바닐라 시럽'], 'quantity': 1, 'unit_price': 5700}] 합계: 5700 원



## HTI 검증 시나리오

| HTI 분류 | 입력 예시 | 기대 결과 |
|---|---|---|
| 메뉴 정보 | 아메리카노 사이즈별 가격 | 정확한 가격·옵션 제시 |
| 취향 추천 | 달콤한 논커피 추천 | 근거가 있는 최대 3개 추천 |
| 식사 페어링 | 식사 메뉴와 어울리는 음료 → 샌드위치 | 실제 판매 음료 3개와 조합 이유 제시 |
| 복합 비교 | 아메리카노와 카푸치노의 맛·가격 | 두 요소 모두 비교 |
| 옵션 주문 | 라떼 M 아이스, 바닐라 시럽 | 지원 옵션 검증 후 장바구니 추가 |
| 수량 변경 | 아아 2잔 주문 → 1잔 취소 | 2→1잔, 9,000→4,500원 |
| 옵션 가격 | 라떼 M + 샷 + 오트밀크 | 서버 단가·합계 6,600원 |
| 짧은 확인 | 네 / 둘 다 / 그걸로 | 직전 사용자 메뉴 문맥 유지와 주문 확정 |
| 모호한 요청 | 평소 먹던 걸로 줘 | 필요한 정보 하나를 재질문 |
| 없는 옵션 | 유자차에 샷 추가 | 지원 불가 안내와 대안 |
| 범위 밖 | 주식 전망 알려줘 | 카페 주문 범위로 정중히 복귀 |
| 검색 실패 | 존재하지 않는 메뉴 | 허위 답변 없이 메뉴 재확인 |

## 피드백 반영

- **서영님:** ‘샌드위치’, ‘치킨’을 없는 판매 메뉴로 판단하던 문제를 수정했다. 두 표현을 식사 페어링 조건으로 해석하고, 카탈로그에 실제로 있는 음료만 추천한다.
- **서진님:** 전체 메뉴판과 가격을 처음부터 표시하고, 추천 이유·사이즈별 가격·수량 취소 후 총액을 함께 검증한다.
- **지희님:** 짧은 후속 답변의 문맥 유지, 옵션 가격 6,600원 일치, ‘네 주문할게요’ 주문 확정을 구현했다.
- 새 피드백은 이름과 내용만 입력하며 최초 상태는 ‘검토 대기’로 저장한다. 상태 이력과 JSON·CSV·Markdown 로그는 제출용 웹 코드에서 관리한다.

## 서버 회귀 테스트 결과

2026-10-12 기준 `npm test` 7/7 통과: 식사 페어링, 2잔 주문 후 1잔 취소, 옵션 포함 6,600원, 짧은 확인 주문 확정, 직전 사용자 메뉴·옵션 복원, ‘둘 다’ 문맥 검색, 15개 메뉴 API.

## 제출 전 점검

- [x] 실제 카페 수준의 음료·디저트 문서를 구성했다.
- [x] 검색 결과를 생성 모델에 제공하는 RAG 흐름을 구현했다.
- [x] 추천·비교·옵션·복합 질문을 처리한다.
- [x] 식사 페어링 질문과 짧은 후속 답변을 처리한다.
- [x] 장바구니 동작을 카탈로그로 재검증한다.
- [x] HTI와 시연 시나리오를 포함한다.
- [x] 범위 밖 질문과 검색 실패를 안전하게 처리한다.
- [x] 서버 회귀 테스트 7개를 통과했다.
- [ ] 제출 직전 전체 셀을 실행해 출력 결과를 저장한다.
